In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../datasets/customer_ml_dataset_eda.csv"
)

In [3]:
df["recency"] = (
    df["average_time_on_site"].max()
    - df["average_time_on_site"]
)

In [4]:
df["frequency"] = df["total_orders"]

In [5]:
df["monetary"] = df["total_spend"]

In [6]:
df["purchase_velocity"] = (
    df["total_orders"] /
    (df["total_sessions"] + 1)
)

In [7]:
df["discount_dependency_score"] = (
    df["average_discount_used"] *
    df["purchase_velocity"]
)

In [8]:
df["return_behavior_score"] = (
    df["cart_abandonment_rate"] * 50
    +
    (1 - df["purchase_velocity"]) * 25
    +
    (1 - (
        df["total_spend"] /
        df["total_spend"].max()
    )) * 25
)

In [9]:
rfm = df[
    [
        "customer_id",
        "recency",
        "frequency",
        "monetary"
    ]
]

In [15]:
rfm["R"] = pd.qcut(
    rfm["recency"],
    q=5,
    labels=False,
    duplicates="drop"
) + 1

rfm["F"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    q=5,
    labels=False,
    duplicates="drop"
) + 1

rfm["M"] = pd.qcut(
    rfm["monetary"],
    q=5,
    labels=False,
    duplicates="drop"
) + 1

In [16]:
rfm["RFM_Score"] = (
    rfm["R"].astype(str)
    +
    rfm["F"].astype(str)
    +
    rfm["M"].astype(str)
)

In [17]:
df = df.merge(
    rfm[
        [
            "customer_id",
            "R",
            "F",
            "M",
            "RFM_Score"
        ]
    ],
    on="customer_id",
    how="left"
)

In [20]:
def segment_customer(score):
    score = str(score)

    if score == "555":
        return "Champions"

    elif score[1:] == "55":
        return "Loyal Customers"

    elif score[0] in ["4", "5"]:
        return "Potential Loyalists"

    elif score[0] in ["1", "2"]:
        return "At Risk"

    return "Regular Customers"

In [21]:
df["churn_risk_score"] = (
    df["cart_abandonment_rate"] * 40
    +
    (1 - df["purchase_velocity"]) * 30
    +
    (
        1 -
        (
            df["customer_satisfaction"] /
            df["customer_satisfaction"].max()
        )
    ) * 30
)

In [22]:
df["high_risk_customer"] = np.where(
    df["churn_risk_score"] > 50,
    1,
    0
)

In [23]:
feature_columns = [
    "recency",
    "frequency",
    "monetary",
    "purchase_velocity",
    "discount_dependency_score",
    "return_behavior_score",
    "R",
    "F",
    "M",
    "churn_risk_score"
]

df[feature_columns].head()

,recency,frequency,monetary,purchase_velocity,discount_dependency_score,return_behavior_score,R,F,M,churn_risk_score
0,744.00,1,1228.72,0.333333,5.833333,64.440879,2,3,2,49.00
1,762.00,0,0.00,0.000000,0.000000,50.000000,2,1,1,36.00
2,1444.33,0,0.00,0.000000,0.000000,83.500000,5,1,1,62.80
3,1062.67,1,1681.96,0.250000,2.917500,40.703183,4,3,2,26.52
4,834.33,0,0.00,0.000000,0.000000,66.500000,3,1,1,49.20


In [24]:
df.to_csv(
    "../datasets/customer_ml_dataset_feature_engineered.csv",
    index=False
)